In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
 
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 4)

In [2]:
OUTPUT_DIR = "Outputs"
anomaly_log = []
 
def log(category, issue, n_affected, action, reason):
    anomaly_log.append({
        "Category"   : category,
        "Issue"      : issue,
        "N_Affected" : n_affected,
        "Action"     : action,
        "Reason"     : reason
    })
    print(f"  [{category}] {issue} → {n_affected} records → {action}")
 

In [3]:
print("\n" + "="*70)
print("LOADING CLEANED DATA (v1)")
print("="*70)
 
loyalty = pd.read_csv(f"{OUTPUT_DIR}/loyalty_cleaned.csv")
flights = pd.read_csv(f"{OUTPUT_DIR}/flights_cleaned.csv")
flights["activity_date"] = pd.to_datetime(flights["activity_date"])
 
# Re-parse date columns saved as strings in CSV (enrollment_year/month stay as integers)
loyalty["enrollment_date"]   = pd.to_datetime(loyalty["enrollment_date"],   errors="coerce")
loyalty["cancellation_date"] = pd.to_datetime(loyalty["cancellation_date"], errors="coerce")
 
ID_COL     = "loyalty_number"
FL_COL     = "total_flights"
DIST_COL   = "distance"
PTS_EARN   = "points_accumulated"
PTS_REDEEM = "points_redeemed"
DOLLAR_PTS = "dollar_cost_points_redeemed"
CLV_COL    = "clv"
YEAR_COL   = "year"
MONTH_COL  = "month"
 
print(f"  Loyalty : {loyalty.shape}")
print(f"  Flights : {flights.shape}")
 
 


LOADING CLEANED DATA (v1)
  Loyalty : (16737, 19)
  Flights : (389065, 10)


In [4]:
loyalty.head()

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month,enrollment_date,cancellation_date,never_flew_flag
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN,2016-02-01,NaT,0
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,Unknown,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN,2016-03-01,NaT,0
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,Unknown,Single,Star,3839.75,Standard,2014,7,2018.0,1.0,2014-07-01,2018-01-01,0
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,Unknown,Single,Star,3839.75,Standard,2013,2,NaN,NaN,2013-02-01,NaT,0
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,NaN,NaN,2014-10-01,NaT,0


In [5]:
print("\n" + "="*70)
print("1. TRAVEL ANOMALIES")
print("="*70)
 
# ── 1a. Flights > 0 but distance = 0 ─────────────────────────────────────────
mask_1a = (flights[FL_COL] > 0) & (flights[DIST_COL] == 0)
n_1a    = mask_1a.sum()
print(f"\n  1a. Flights booked but zero distance : {n_1a:,} rows")
if n_1a > 0:
    print(flights[mask_1a][[ID_COL, YEAR_COL, MONTH_COL,
                              FL_COL, DIST_COL]].head(10).to_string())
    flights.loc[mask_1a, DIST_COL] = np.nan   # mark as unknown, not 0
    log("Travel", "Flights > 0 but distance = 0", n_1a,
        "Set distance to NaN",
        "Zero distance with positive flight count is physically impossible; "
        "NaN is more honest than 0 for downstream aggregation")
 
# ── 1b. Distance > 0 but flights = 0 ─────────────────────────────────────────
mask_1b = (flights[DIST_COL] > 0) & (flights[FL_COL] == 0)
n_1b    = mask_1b.sum()
print(f"\n  1b. Distance > 0 but zero flights    : {n_1b:,} rows")
if n_1b > 0:
    print(flights[mask_1b][[ID_COL, YEAR_COL, MONTH_COL,
                              FL_COL, DIST_COL]].head(10).to_string())
    flights.loc[mask_1b, DIST_COL] = 0
    log("Travel", "Distance > 0 but flights = 0", n_1b,
        "Set distance to 0",
        "Cannot accrue distance without a flight; likely points-only transaction")
 
# ── 1c. Unusually high flights per month ─────────────────────────────────────
# Max realistic: ~30 flights/month for a very frequent flyer
fl_99   = flights[FL_COL].quantile(0.99)
fl_max  = flights[FL_COL].max()
mask_1c = flights[FL_COL] > 30
n_1c    = mask_1c.sum()
print(f"\n  1c. Flights per month > 30           : {n_1c:,} rows  "
      f"(99th pct={fl_99:.0f}, max={fl_max})")
if n_1c > 0:
    print(flights[mask_1c][[ID_COL, YEAR_COL, MONTH_COL, FL_COL]].to_string())
    flights.loc[mask_1c, FL_COL] = 30
    log("Travel", "Monthly flights > 30", n_1c,
        "Capped at 30",
        "30 flights/month is ~1 per day; beyond this is likely a data entry error")
 
# ── 1d. Distance per flight — check average km per flight ────────────────────
flights["dist_per_flight"] = np.where(
    flights[FL_COL] > 0,
    flights[DIST_COL] / flights[FL_COL],
    np.nan
)
# Realistic range: 200 km (short hop) to 15,000 km (ultra long haul)
mask_1d_low  = (flights["dist_per_flight"] < 100) & (flights[FL_COL] > 0)
mask_1d_high = flights["dist_per_flight"] > 20000
n_1d_low     = mask_1d_low.sum()
n_1d_high    = mask_1d_high.sum()
 
print(f"\n  1d. Avg distance per flight < 100 km : {n_1d_low:,} rows")
print(f"      Avg distance per flight > 20,000 km: {n_1d_high:,} rows")
print(f"      Distance per flight — percentiles:")
print(flights["dist_per_flight"].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]
).round(1).to_string())
 
if n_1d_low > 0:
    log("Travel", "Avg distance/flight < 100 km", n_1d_low,
        "Flagged with 'low_dist_flag'; kept",
        "Could be shuttle/regional flights; not impossible but suspicious")
    flights["low_dist_flag"] = mask_1d_low.astype(int)
 
if n_1d_high > 0:
    log("Travel", "Avg distance/flight > 20,000 km", n_1d_high,
        "Capped distance at 20,000 km per flight × flight count",
        "Longest commercial route is ~18,000 km; beyond is a data error")
    flights.loc[mask_1d_high, DIST_COL] = 20000 * flights.loc[mask_1d_high, FL_COL]
 
# Drop helper column
flights.drop(columns=["dist_per_flight"], inplace=True)
 


1. TRAVEL ANOMALIES

  1a. Flights booked but zero distance : 0 rows

  1b. Distance > 0 but zero flights    : 0 rows

  1c. Flights per month > 30           : 2 rows  (99th pct=6, max=32)
        loyalty_number  year  month  total_flights
273780          732304  2018      6             32
273782          732304  2018      8             32
  [Travel] Monthly flights > 30 → 2 records → Capped at 30

  1d. Avg distance per flight < 100 km : 0 rows
      Avg distance per flight > 20,000 km: 0 rows
      Distance per flight — percentiles:
count    177973.0
mean       1498.2
std         577.4
min         500.0
1%          520.0
5%          599.0
25%         999.0
50%        1498.0
75%        1998.0
95%        2400.0
99%        2480.3
max        2500.0


In [6]:
print("\n" + "="*70)
print("2. POINTS ANOMALIES")
print("="*70)
 
# ── 2a. Points earned per flight — check ratio ────────────────────────────────
# Typical: 5–20 points per km, so points ~ distance * 5 to 20
flights["pts_per_flight"] = np.where(
    flights[FL_COL] > 0,
    flights[PTS_EARN] / flights[FL_COL],
    np.nan
)
print(f"\n  2a. Points earned per flight:")
print(flights["pts_per_flight"].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]
).round(1).to_string())
 
# Extreme outliers: top 0.1%
pts_999 = flights["pts_per_flight"].quantile(0.999)
mask_2a  = flights["pts_per_flight"] > pts_999 * 3
n_2a     = mask_2a.sum()
print(f"\n  Points/flight > 3x the 99.9th pct ({pts_999*3:.0f}): {n_2a:,} rows")
if n_2a > 0:
    log("Points", f"Points per flight > {pts_999*3:.0f} (3x 99.9th pct)", n_2a,
        "Flagged; kept in dataset",
        "Could be bonus/promo points; removing may lose valid promotions")
    flights["pts_earn_outlier_flag"] = mask_2a.astype(int)
flights.drop(columns=["pts_per_flight"], inplace=True)
 
# ── 2b. Points redeemed with zero points earned (same month) ─────────────────
mask_2b = (flights[PTS_REDEEM] > 0) & (flights[PTS_EARN] == 0)
n_2b    = mask_2b.sum()
print(f"\n  2b. Redemption with zero earnings (same month): {n_2b:,} rows")
# This is OK — members redeem from accumulated balance, not same-month earnings
print("      (Expected — members redeem from balance, not current month earnings)")
if n_2b > 0:
    log("Points", "Redemption > 0 with same-month earnings = 0", n_2b,
        "Kept as-is",
        "Redemption draws from accumulated balance; "
        "same-month zero earning is perfectly valid")
 
# ── 2c. Dollar cost vs points redeemed consistency ───────────────────────────
# Check if dollar_cost_points_redeemed is proportional to points_redeemed
redeem_rows = flights[flights[PTS_REDEEM] > 0].copy()
if len(redeem_rows) > 0:
    redeem_rows["cost_per_point"] = (
        redeem_rows[DOLLAR_PTS] / redeem_rows[PTS_REDEEM]
    )
    print(f"\n  2c. Dollar cost per point redeemed:")
    print(redeem_rows["cost_per_point"].describe(
        percentiles=[.01, .05, .25, .5, .75, .95, .99]
    ).round(4).to_string())
 
    # Flag extreme outliers (> 10× median)
    med_cpp  = redeem_rows["cost_per_point"].median()
    mask_2c  = redeem_rows["cost_per_point"] > med_cpp * 10
    n_2c     = mask_2c.sum()
    print(f"\n  Cost/point > 10x median ({med_cpp*10:.4f}): {n_2c:,} rows")
    if n_2c > 0:
        log("Points",
            f"Dollar cost per point > 10x median ({med_cpp:.4f})", n_2c,
            "Flagged; kept",
            "Could be premium redemptions or data entry errors; "
            "not worth dropping without further investigation")
 
 
# ── 2d. Lifetime redemption > lifetime earnings (member level) ────────────────
member_pts = (flights.groupby(ID_COL)[[PTS_EARN, PTS_REDEEM]].sum())
over_redeem = member_pts[member_pts[PTS_REDEEM] > member_pts[PTS_EARN]]
n_2d = len(over_redeem)
print(f"\n  2d. Members with lifetime redemptions > lifetime earnings: {n_2d:,}")
if n_2d > 0:
    excess = (over_redeem[PTS_REDEEM] - over_redeem[PTS_EARN]).describe().round(0)
    print(f"  Excess redeemed (points):\n{excess.to_string()}")
    log("Points", "Lifetime redeemed > lifetime earned", n_2d,
        "Flagged with 'over_redeem_flag' in loyalty table",
        "Possible partner transfers or system migration artefacts; "
        "kept to avoid losing valid members")
    loyalty["over_redeem_flag"] = loyalty[ID_COL].isin(
        over_redeem.index
    ).astype(int)
 
 


2. POINTS ANOMALIES

  2a. Points earned per flight:
count    177973.0
mean       1521.2
std         602.6
min         500.0
1%          520.0
5%          602.0
25%        1009.0
50%        1513.0
75%        2019.0
95%        2425.0
99%        2766.4
max        3750.0

  Points/flight > 3x the 99.9th pct (10976): 0 rows

  2b. Redemption with zero earnings (same month): 2,918 rows
      (Expected — members redeem from balance, not current month earnings)
  [Points] Redemption > 0 with same-month earnings = 0 → 2918 records → Kept as-is

  2c. Dollar cost per point redeemed:
count    23873.0000
mean         0.1800
std          0.0006
min          0.1785
1%           0.1788
5%           0.1791
25%          0.1795
50%          0.1800
75%          0.1805
95%          0.1810
99%          0.1813
max          0.1815

  Cost/point > 10x median (1.8004): 0 rows

  2d. Members with lifetime redemptions > lifetime earnings: 11
  Excess redeemed (points):
count     11.0
mean     398.0
std      17

In [7]:
print("\n" + "="*70)
print("3. CLV ANOMALIES")
print("="*70)
 
# ── 3a. CLV distribution ──────────────────────────────────────────────────────
print(f"\n  3a. CLV descriptive stats:")
print(loyalty[CLV_COL].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]
).round(2).to_string())
 
# ── 3b. CLV outliers (top 1%) ─────────────────────────────────────────────────
clv_99 = loyalty[CLV_COL].quantile(0.99)
n_3b   = (loyalty[CLV_COL] > clv_99).sum()
print(f"\n  3b. CLV above 99th percentile (>{clv_99:.0f}): {n_3b:,} members")
log("CLV", f"CLV > 99th pct ({clv_99:.0f})", n_3b,
    "Flagged with 'high_clv_flag'; not capped",
    "Top CLV members are genuinely valuable; capping would distort segmentation")
loyalty["high_clv_flag"] = (loyalty[CLV_COL] > clv_99).astype(int)
 
# ── 3c. CLV vs flight activity mismatch ───────────────────────────────────────
# Members in top 25% CLV but bottom 25% flight activity = suspicious
member_total_flights = flights.groupby(ID_COL)[FL_COL].sum().reset_index()
loyalty = loyalty.merge(member_total_flights.rename(
    columns={FL_COL: "total_flights_ever"}), on=ID_COL, how="left"
)
loyalty["total_flights_ever"] = loyalty["total_flights_ever"].fillna(0)
 
clv_75  = loyalty[CLV_COL].quantile(0.75)
fl_25   = loyalty["total_flights_ever"].quantile(0.25)
mask_3c = (loyalty[CLV_COL] > clv_75) & (loyalty["total_flights_ever"] <= fl_25)
n_3c    = mask_3c.sum()
print(f"\n  3c. High CLV (top 25%) but low flights (bottom 25%): {n_3c:,} members")
if n_3c > 0:
    print("      Sample:")
    print(loyalty[mask_3c][[ID_COL, CLV_COL, "total_flights_ever"]].head(8).to_string())
    log("CLV", "High CLV but low flight activity", n_3c,
        "Flagged with 'clv_activity_mismatch_flag'",
        "CLV may include non-flight spend (partner programs, credit card). "
        "These members need separate treatment in segmentation")
    loyalty["clv_activity_mismatch_flag"] = mask_3c.astype(int)
 
 


3. CLV ANOMALIES

  3a. CLV descriptive stats:
count    16737.00
mean      7988.90
std       6860.98
min       1898.01
1%        2230.45
5%        2475.51
25%       3980.84
50%       5780.18
75%       8940.58
95%      22031.69
99%      35928.64
max      83325.38

  3b. CLV above 99th percentile (>35929): 168 members
  [CLV] CLV > 99th pct (35929) → 168 records → Flagged with 'high_clv_flag'; not capped

  3c. High CLV (top 25%) but low flights (bottom 25%): 1,084 members
      Sample:
      loyalty_number       clv  total_flights_ever
981           376929   9753.31                   3
2047          961264  19688.32                   0
2805          321734   8973.11                  19
2809          847283   8992.78                   0
2815          726544   9009.22                  18
2819          223313   9015.92                  18
2820          924796   9022.08                   0
2824          288636   9029.71                  19
  [CLV] High CLV but low flight activity → 1084 re

In [8]:
print("\n" + "="*70)
print("4. DEMOGRAPHIC & ENROLLMENT ANOMALIES")
print("="*70)
 
# ── 4a. Enrollment year range ─────────────────────────────────────────────────
print(f"\n  4a. Enrollment year range:")
print(loyalty["enrollment_year"].value_counts().sort_index().to_string())
mask_4a = (loyalty["enrollment_year"] < 2000) | (loyalty["enrollment_year"] > 2018)
n_4a    = mask_4a.sum()
if n_4a > 0:
    print(f"\n  Enrollment year outside 2000-2018: {n_4a:,}")
    log("Demographics", "Enrollment year outside 2000-2018", n_4a,
        "Flagged; kept",
        "May be legacy members from predecessor program")
 
# ── 4b. Enrollment month validity ────────────────────────────────────────────
mask_4b = ~loyalty["enrollment_month"].between(1, 12)
n_4b    = mask_4b.sum()
print(f"\n  4b. Invalid enrollment month (not 1-12): {n_4b:,}")
if n_4b > 0:
    log("Demographics", "Invalid enrollment month", n_4b,
        "Set to NaN",
        "Month outside 1-12 is impossible")
    loyalty.loc[mask_4b, "enrollment_month"] = np.nan
 
# ── 4c. Members who flew BEFORE enrollment ───────────────────────────────────
# enrollment_date already correctly parsed at load time — use directly
 
first_flight = (flights[flights[FL_COL] > 0]
                .groupby(ID_COL)["activity_date"]
                .min()
                .reset_index()
                .rename(columns={"activity_date": "first_flight_date"}))
 
loyalty = loyalty.merge(first_flight, on=ID_COL, how="left")
mask_4c = (
    loyalty["first_flight_date"].notna() &
    loyalty["enrollment_date"].notna() &
    (loyalty["first_flight_date"] < loyalty["enrollment_date"])
)
n_4c = mask_4c.sum()
print(f"\n  4c. Members who flew before enrollment date: {n_4c:,}")
if n_4c > 0:
    print(loyalty[mask_4c][
        [ID_COL, "enrollment_date", "first_flight_date"]
    ].head(10).to_string())
    log("Demographics", "First flight before enrollment date", n_4c,
        "Flagged; enrollment date set to first_flight_date for these members",
        "Flight records pre-date enrollment; likely data migration issue. "
        "Using first flight date as proxy enrollment date.")
    loyalty.loc[mask_4c, "enrollment_date"] = loyalty.loc[
        mask_4c, "first_flight_date"
    ]
 


4. DEMOGRAPHIC & ENROLLMENT ANOMALIES

  4a. Enrollment year range:
enrollment_year
2012    1686
2013    2397
2014    2370
2015    2331
2016    2456
2017    2487
2018    3010

  4b. Invalid enrollment month (not 1-12): 0

  4c. Members who flew before enrollment date: 498
     loyalty_number enrollment_date first_flight_date
59           864324      2018-06-01        2017-02-01
132          366540      2018-03-01        2018-02-01
177          645919      2018-03-01        2018-02-01
222          670686      2018-05-01        2017-01-01
234          342483      2018-03-01        2018-02-01
237          816881      2018-06-01        2017-01-01
330          120613      2018-12-01        2017-02-01
339          947876      2018-12-01        2017-01-01
373          696577      2018-04-01        2018-02-01
376          450979      2018-11-01        2017-01-01
  [Demographics] First flight before enrollment date → 498 records → Flagged; enrollment date set to first_flight_date for these mem

In [9]:
print("\n" + "="*70)
print("5. TEMPORAL ANOMALIES")
print("="*70)
 
# Members who have a cancellation date — did they fly after it?
cancelled = loyalty[loyalty["cancellation_year"].notna()].copy()
 
if len(cancelled) > 0:
    # cancellation_date already parsed at load time — use directly
    # Merge with flights
    cancelled_flights = flights.merge(
        cancelled[[ID_COL, "cancellation_date"]], on=ID_COL, how="inner"
    )
    mask_5a = (
        cancelled_flights[FL_COL] > 0) & (
        cancelled_flights["activity_date"] > cancelled_flights["cancellation_date"]
    )
    n_5a = mask_5a.sum()
    affected_members = cancelled_flights[mask_5a][ID_COL].nunique()
    print(f"\n  5a. Flight records AFTER cancellation date:")
    print(f"      Rows: {n_5a:,}  |  Members: {affected_members:,}")
 
    if n_5a > 0:
        # Zero out flights after cancellation (treat as ghost records)
        post_cancel_ids = set(
            zip(cancelled_flights[mask_5a][ID_COL],
                cancelled_flights[mask_5a]["activity_date"])
        )
        def is_post_cancel(row):
            return (row[ID_COL], row["activity_date"]) in post_cancel_ids
 
        # Flag these rows in flights
        flights = flights.merge(
            cancelled[[ID_COL, "cancellation_date"]], on=ID_COL, how="left"
        )
        mask_post = (
            flights["cancellation_date"].notna() &
            (flights["activity_date"] > flights["cancellation_date"]) &
            (flights[FL_COL] > 0)
        )
        flights.loc[mask_post, [FL_COL, DIST_COL, PTS_EARN,
                                  PTS_REDEEM, DOLLAR_PTS]] = 0
        flights.drop(columns=["cancellation_date"], inplace=True)
 
        log("Temporal", "Flights recorded after member cancellation",
            n_5a, "Zeroed out flight metrics for post-cancellation rows",
            "A cancelled member cannot earn points or fly under this program; "
            "these are system lag records")
    else:
        if "cancellation_date" in flights.columns:
            flights.drop(columns=["cancellation_date"], inplace=True)
 
# ── 5b. Check zero-flight months ratio ───────────────────────────────────────
total_rows        = len(flights)
zero_flight_rows  = (flights[FL_COL] == 0).sum()
print(f"\n  5b. Zero-flight monthly records: "
      f"{zero_flight_rows:,} / {total_rows:,} "
      f"({zero_flight_rows/total_rows*100:.1f}%)")
print("      (Expected — most members don't fly every month)")
 


5. TEMPORAL ANOMALIES

  5a. Flight records AFTER cancellation date:
      Rows: 0  |  Members: 0

  5b. Zero-flight monthly records: 211,092 / 389,065 (54.3%)
      (Expected — most members don't fly every month)


In [10]:
print("\n" + "="*70)
print("6. STATISTICAL OUTLIER SUMMARY")
print("="*70)
 
numeric_cols = [FL_COL, DIST_COL, PTS_EARN, PTS_REDEEM, DOLLAR_PTS]
print(f"\n  {'Column':<35} {'Min':>10} {'P1':>10} {'P99':>10} {'Max':>10} {'Outliers(>3σ)':>15}")
print("  " + "-"*90)
for col in numeric_cols:
    col_data  = flights[col].replace(0, np.nan).dropna()
    mean, std = col_data.mean(), col_data.std()
    p1        = col_data.quantile(0.01)
    p99       = col_data.quantile(0.99)
    n_out     = (col_data > mean + 3*std).sum()
    print(f"  {col:<35} {col_data.min():>10.1f} {p1:>10.1f} "
          f"{p99:>10.1f} {col_data.max():>10.1f} {n_out:>15,}")
 


6. STATISTICAL OUTLIER SUMMARY

  Column                                     Min         P1        P99        Max   Outliers(>3σ)
  ------------------------------------------------------------------------------------------
  total_flights                              1.0        1.0        9.0       30.0           1,883
  distance                                 500.0      580.0    14916.0    67284.0           1,695
  points_accumulated                       500.0      581.0    20700.0   100926.0           2,020
  points_redeemed                          289.0      309.0      823.0      996.0               4
  dollar_cost_points_redeemed               52.0       56.0      148.0      179.0               4


In [11]:
print("\n" + "="*70)
print("7. SAVING ANOMALY PLOTS")
print("="*70)
 
# Plot 1: Distance per flight distribution
valid_dist = flights[flights[FL_COL] > 0][DIST_COL] / flights[flights[FL_COL] > 0][FL_COL]
fig, ax = plt.subplots()
valid_dist.clip(upper=valid_dist.quantile(0.99)).plot(
    kind="hist", bins=60, ax=ax, color="steelblue", edgecolor="white"
)
ax.set_title("Distribution of Average Distance per Flight (km)", fontweight="bold")
ax.set_xlabel("km per flight")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_anomaly_dist_per_flight.png", dpi=120)
plt.close()
print(f"  Saved: plot_anomaly_dist_per_flight.png")
 
# Plot 2: Points earned per flight
valid_pts = flights[flights[FL_COL] > 0][PTS_EARN] / flights[flights[FL_COL] > 0][FL_COL]
fig, ax = plt.subplots()
valid_pts.clip(upper=valid_pts.quantile(0.99)).plot(
    kind="hist", bins=60, ax=ax, color="coral", edgecolor="white"
)
ax.set_title("Distribution of Points Earned per Flight", fontweight="bold")
ax.set_xlabel("Points per flight")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_anomaly_pts_per_flight.png", dpi=120)
plt.close()
print(f"  Saved: plot_anomaly_pts_per_flight.png")
 
# Plot 3: CLV vs total flights scatter
fig, ax = plt.subplots(figsize=(8, 5))
sample = loyalty.sample(min(2000, len(loyalty)), random_state=42)
ax.scatter(sample["total_flights_ever"], sample[CLV_COL],
           alpha=0.3, s=10, color="steelblue")
ax.set_title("CLV vs Total Flights (sample of 2,000)", fontweight="bold")
ax.set_xlabel("Total Flights (2017–2018)")
ax.set_ylabel("CLV")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_anomaly_clv_vs_flights.png", dpi=120)
plt.close()
print(f"  Saved: plot_anomaly_clv_vs_flights.png")
 


7. SAVING ANOMALY PLOTS
  Saved: plot_anomaly_dist_per_flight.png
  Saved: plot_anomaly_pts_per_flight.png
  Saved: plot_anomaly_clv_vs_flights.png


In [12]:
print("\n" + "="*70)
print("8. SAVING CLEANED V2 DATA")
print("="*70)
 
# Drop helper columns added during audit
drop_loyalty = ["total_flights_ever", "first_flight_date"]
loyalty.drop(columns=[c for c in drop_loyalty if c in loyalty.columns],
             inplace=True)
 
loyalty.to_csv(f"{OUTPUT_DIR}/loyalty_cleaned_v2.csv", index=False)
flights.to_csv(f"{OUTPUT_DIR}/flights_cleaned_v2.csv", index=False)
 
print(f"  Saved: loyalty_cleaned_v2.csv  {loyalty.shape}")
print(f"  Saved: flights_cleaned_v2.csv  {flights.shape}")
 
# ── Anomaly Report ────────────────────────────────────────────────────────────
report = pd.DataFrame(anomaly_log)
report.to_csv(f"{OUTPUT_DIR}/anomaly_report.csv", index=False)
print(f"\n  Saved: anomaly_report.csv  ({len(anomaly_log)} anomalies documented)")
print(f"\n  Full Anomaly Report:")
print(report.to_string(index=False))
 
print(f"""
{"="*70}
PHASE 1B COMPLETE
{"="*70}
 
  Use these files going forward:
    loyalty_cleaned_v2.csv
    flights_cleaned_v2.csv
 
  Update phase2_feature_engineering.py to load _v2 files.
  Run: python phase2_feature_engineering.py  (after updating file paths)
""")


8. SAVING CLEANED V2 DATA
  Saved: loyalty_cleaned_v2.csv  (16737, 22)
  Saved: flights_cleaned_v2.csv  (389065, 10)

  Saved: anomaly_report.csv  (6 anomalies documented)

  Full Anomaly Report:
    Category                                       Issue  N_Affected                                                              Action                                                                                                                  Reason
      Travel                        Monthly flights > 30           2                                                        Capped at 30                                                30 flights/month is ~1 per day; beyond this is likely a data entry error
      Points Redemption > 0 with same-month earnings = 0        2918                                                          Kept as-is                                   Redemption draws from accumulated balance; same-month zero earning is perfectly valid
      Points         Lifetime re